In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout, BatchNormalization, Activation
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, auc
import numpy as np
import os
import shutil
import seaborn as sns
import matplotlib.pyplot as plt

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# Constants — update DATASET_PATH to point to your local dataset
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.00005
SPLIT_SEED = 42

DATASET_PATH = "data/lumpydataset"          # <-- update this
SPLIT_DATASET_PATH = "data/split"           # <-- update this
CLASS_NAMES = ['LSD_Infected', 'Healthy']

In [ ]:
# Function to split dataset
def split_dataset(original_path, split_path, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1):
    # Skip if split already exists
    if os.path.exists(split_path):
        print(f"Split dataset already exists at '{split_path}'. Skipping split.")
        return

    os.makedirs(split_path)

    for category in os.listdir(original_path):
        category_path = os.path.join(original_path, category)
        if not os.path.isdir(category_path):
            continue

        images = os.listdir(category_path)
        train, temp = train_test_split(images, test_size=(1 - train_ratio), random_state=SPLIT_SEED)
        val, test = train_test_split(temp, test_size=(test_ratio / (val_ratio + test_ratio)), random_state=SPLIT_SEED)

        for dataset_type, dataset in zip(['train', 'val', 'test'], [train, val, test]):
            dataset_category_path = os.path.join(split_path, dataset_type, category)
            os.makedirs(dataset_category_path, exist_ok=True)
            for img_file in dataset:
                shutil.copy(os.path.join(category_path, img_file), os.path.join(dataset_category_path, img_file))

    print("Dataset split complete.")

split_dataset(DATASET_PATH, SPLIT_DATASET_PATH)

In [ ]:
# Data Augmentation & Loading
datagen_train = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

datagen_val_test = ImageDataGenerator(rescale=1.0/255.0)

train_generator = datagen_train.flow_from_directory(
    os.path.join(SPLIT_DATASET_PATH, 'train'),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = datagen_val_test.flow_from_directory(
    os.path.join(SPLIT_DATASET_PATH, 'val'),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = datagen_val_test.flow_from_directory(
    os.path.join(SPLIT_DATASET_PATH, 'test'),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
# Load Pretrained ResNet50 Model
resnet_base = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
resnet_base.trainable = False  # Freeze all layers initially

In [ ]:
# Fine-tune last 50 layers
for layer in resnet_base.layers[-50:]:
    layer.trainable = True

In [ ]:
# Define Model Architecture
inputs = Input(shape=(224, 224, 3))
x = resnet_base(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.4)(x)
x = Dense(256, kernel_regularizer=keras.regularizers.l2(0.001))(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(len(CLASS_NAMES), activation='softmax')(x)

model = Model(inputs, outputs)

In [ ]:
# Compile Model
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
checkpoint = ModelCheckpoint("best_cattle_model.keras", save_best_only=True, monitor="val_accuracy")
lr_scheduler = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1)

In [ ]:
# Train Model
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[early_stop, checkpoint, lr_scheduler]
)

In [ ]:
# Save Model (use .keras format, not legacy .h5)
model.save("cattle_resnet_lsd_model.keras")

In [ ]:
# Plot Accuracy
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Accuracy per Epoch')
plt.show()

# Plot Loss
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Loss per Epoch')
plt.show()

In [ ]:
# Validation result
eval_result = model.evaluate(val_generator)
print(f"Validation Loss: {eval_result[0]:.4f}, Validation Accuracy: {eval_result[1]:.4f}")

In [ ]:
# Test result
test_result = model.evaluate(test_generator)
print(f"Test Loss: {test_result[0]:.4f}, Test Accuracy: {test_result[1]:.4f}")

In [ ]:
# Compute Metrics
y_true = test_generator.classes
y_pred_probs = model.predict(test_generator)
y_pred = np.argmax(y_pred_probs, axis=1)

# For binary classification, use probability of the positive class (index 1)
# not np.max(), which gives the wrong scores for the negative class
y_scores = y_pred_probs[:, 1]

accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='weighted')
auc_roc = roc_auc_score(y_true, y_scores)

precision, recall, _ = precision_recall_curve(y_true, y_scores)
pr_auc = auc(recall, precision)

print(f"Model Accuracy:              {accuracy:.4f}")
print(f"Model F1 Score:              {f1:.4f}")
print(f"Model AUC-ROC:               {auc_roc:.4f}")
print(f"Model Precision-Recall AUC:  {pr_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
# Confusion Matrix
conf_matrix = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()